In [1]:
from collections import defaultdict
import os
import pickle
import re
import sys
from collections.abc import Iterable

import numpy as np
import pandas as pd

from IPython.core.display import display, HTML

In [2]:
comp_property_pii_df = pd.read_csv('../data/combined_property_interglad.csv')

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3258: DtypeWarning: Columns (1074,1075) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [3]:
comp_property_pii_df

,SiO2_x,Li2O_x,Na2O_x,K2O_x,Al2O3_x,MgO_x,CaO_x,B2O3_x,BaO_x,Fe2O3_x,...,PATENT,NOTE,COMMERCIAL_GLASS,LINK_COMP,GFR_FLAG,VOL,PAGE,PII,DOI,JOURNAL
0,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
1,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
2,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
3,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
4,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.75,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0
12959,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.75,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0
12960,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.50,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0
12961,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.50,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0


In [4]:
comp_property_pii_df.drop_duplicates()

,SiO2_x,Li2O_x,Na2O_x,K2O_x,Al2O3_x,MgO_x,CaO_x,B2O3_x,BaO_x,Fe2O3_x,...,PATENT,NOTE,COMMERCIAL_GLASS,LINK_COMP,GFR_FLAG,VOL,PAGE,PII,DOI,JOURNAL
0,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
1,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
2,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
3,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
4,85.0,0.0,0.0,15.0,0.0,0.0,0.0,0.00,0.0,0.0,...,NaN,Effect of high pressure on elastic constant,NaN,NaN,NaN,13.0,399.0,0022309374900039,doi:10.1016/0022-3093(74)90003-9,207.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.75,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0
12959,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.75,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0
12960,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.50,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0
12961,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.50,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,351.0,2166.0,S0022309305003959,doi:10.1016/j.jnoncrysol.2005.06.004,207.0


In [5]:
comp_property_pii_df.keys()

Index(['SiO2_x', 'Li2O_x', 'Na2O_x', 'K2O_x', 'Al2O3_x', 'MgO_x', 'CaO_x',
       'B2O3_x', 'BaO_x', 'Fe2O3_x',
       ...
       'PATENT', 'NOTE', 'COMMERCIAL_GLASS', 'LINK_COMP', 'GFR_FLAG', 'VOL',
       'PAGE', 'PII', 'DOI', 'JOURNAL'],
      dtype='object', length=1083)

In [6]:
comp_property_pii_df["GLASS_ID"].nunique()
comp_property_pii_df["GLASS_ID"]

0         54781
1         54781
2         54781
3         54781
4         54781
          ...  
12958    266972
12959    266972
12960    266973
12961    266973
12962    266973
Name: GLASS_ID, Length: 12963, dtype: int64

In [7]:
prop_id_counts = comp_property_pii_df['PROP_ID'].value_counts()
#print(prop_id_counts)
prop_id_counts_list = prop_id_counts.index.tolist()
print(prop_id_counts_list)

[1140, 510, 1014, 2010, 1020, 1011, 180, 540, 3012, 1118, 3174, 50, 1116, 60, 160, 70, 1603, 2210, 2051, 1119, 120, 1633, 2209, 102, 1305, 1123, 2200, 2212, 1390, 1306]


In [8]:
#pii_ec_set = set()
comp_ec = comp_property_pii_df[comp_property_pii_df["PROP_ID"] == 160]
#comp_ec
pii_ec_set = comp_ec['PII'].unique()
pii_ec_set

array(['S0022309397003372', 'S0022309399003488', 'S0022309399004846',
       'S0022309399006377', 'S002230930000082X', 'S002230930200950X',
       'S0022309302019348', 'S0022309302019373', 'S0022309302019452',
       'S0022309302019488', 'S0022309304004910', 'S0022309304004934',
       'S0022309304004971'], dtype=object)

In [9]:
table_dir = '../data/' #applying distant supv on the discomat data
save_data = '../data/data_kausik_max_discomat/'

# comp_data = pickle.load(open(os.path.join(table_dir, 'train_data.pkl'), 'rb')) + \
#             pickle.load(open(os.path.join(table_dir, 'val_test_data.pkl'), 'rb'))
comp_train_data = pickle.load(open(os.path.join(table_dir, 'train_data.pkl'), 'rb'))
comp_data_dict = {(c['pii'], c['t_idx']): c for c in comp_train_data}
train_val_test_split = pickle.load(open(os.path.join(table_dir, 'train_val_test_split.pkl'), 'rb'))
splits = ['train', 'val', 'test']
print(len(comp_data_dict))
print(len(train_val_test_split['train']), len(train_val_test_split['val']), len(train_val_test_split['test']))

4408
4408 738 737


In [10]:
pii_tables = defaultdict(list)
for pii_tid in train_val_test_split['val']:
    pii_tables[pii_tid[0]] = []
for pii_tid in train_val_test_split['test']:
    pii_tables[pii_tid[0]] = []
for pii_tid in comp_data_dict:
    pii_tables[pii_tid[0]].append(comp_data_dict[pii_tid])

In [11]:
#pii_tables['S0022309307006989']

In [12]:
print(len(comp_property_pii_df['PII'].unique()))

617


In [13]:
comp_property_pii_df[comp_property_pii_df["PII"] == 'S0022309307006989']

,SiO2_x,Li2O_x,Na2O_x,K2O_x,Al2O3_x,MgO_x,CaO_x,B2O3_x,BaO_x,Fe2O3_x,...,PATENT,NOTE,COMMERCIAL_GLASS,LINK_COMP,GFR_FLAG,VOL,PAGE,PII,DOI,JOURNAL


In [14]:
num_pattern = re.compile(r'\d*\.\d+|\d+')
re.findall(num_pattern, '1.1gm/cm')

['1.1']

In [15]:
# def find_num(string):
#     #e.g. already in int or float form: 12.5 -> 12.5
#     try:
#         return float(string)
#     except:
#         pass
#     #e.g. 12.5 - 13.5 -> 12.5
#     range_regex = re.compile('\d+\.?\d*\s*-\s*\d+\.?\d*')
#     try:
#         ranges = range_regex.search(string).group().split('-')
#         num = float(ranges[0])
#         return num
#     except:
#         pass
#     #e.g. 12.2 (5.2) -> 12.2
#     bracket_regex = re.compile('(\d+\.?\d*)\s*\(\d*.?\d*\)')
#     try:
#         extracted_value = float(bracket_regex.search(string).group(1))
#         return float(extracted_value)
#     except:
#         pass
#     #e.g. 12.3 ± 0.5 -> 12.3
#     plusmin_regex = re.compile('(\d+\.?\d*)(\s*[±+-]+\s*\d+\.?\d*)')  
#     try:
#         extracted_value = float(plusmin_regex.search(string).group(1))
#         return extracted_value
#     except AttributeError:
#         pass
#     #e.g. <0.05 -> 0.05  |  >72.0 -> 72.0    | ~12 -> 12
#     lessthan_roughly_regex = re.compile('([<]|[~]|[>])=?\s*\d+\.*\d*')
#     try:
#         extracted_value = lessthan_roughly_regex.search(string).group()
#         num_regex = re.compile('\d+\.*\d*')
#         extracted_value = num_regex.search(extracted_value).group()
#         return float(extracted_value)
#     except:
#         pass
#     # e.g. 0.4:0.6 (ratios)
#     if ':' in string:
#         split = string.split(":")
#         try:
#             extracted_value = round(float(split[0])/float(split[1]), 3)
#             return extracted_value
#         except:
#             pass
#     return None

In [16]:
def find_num(string):
    #remove tabs or unnecessary spaces
    try:
        string = string.strip()
    except:
        pass
    #e.g. already in int or float form: 12.5 -> 12.5
    try:
        if string[0] == '-':
            return float(string[1:])*(-1)
        else:
            return float(string)
    except:
        pass
    # e.g. 99.1x10-7
    range_regex = re.compile('^\d+\.?\d*\s*x\s*\d+\.?\d*[-|+]?\s*\d+\.?\d*$')
    try:
#         print('alla')
        ranges = range_regex.search(string).group().split('x')
        if '-' in ranges[1]:
            numu = ranges[1].split('-')
#             print(ranges)
#             print(float(numu[0]), float(numu[1]))
            num = float(ranges[0]) * pow(float(numu[0]), (-float(numu[1])))
            formatted_result = "{:.3e}".format(num)
            return formatted_result
        elif '+' in ranges[1]:
            numu = ranges[1].split('+')
            num = float(ranges[0]) * pow(float(numu[0]), (float(numu[1])))
#             print(num)
            return num
        num = float(ranges[0]) * float(ranges[1])
        formatted_result = "{:.3e}".format(num)
        return formatted_result
    except:
        pass
    #e.g. 12.5 - 13.5 -> 12.5
    range_regex = re.compile('^\d+\.?\d*\s*-\s*\d+\.?\d*')
    try:
        ranges = range_regex.search(string).group().split('-')
        num = float(ranges[0])
#         print(num)
        return num
    except:
        pass
#     print('alla')
    #e.g. 12.2 (5.2) -> 12.2
    bracket_regex = re.compile('(\d+\.?\d*)\s*\(\d*.?\d*\)')
    try:
        extracted_value = float(bracket_regex.search(string).group(1))
        return float(extracted_value)
    except:
        pass
    #e.g. 12.3 ± 0.5 -> 12.3
    plusmin_regex = re.compile('^(\d+\.?\d*)(\s*[±+-]+\s*\d+\.?\d*)')
    try:
        extracted_value = float(plusmin_regex.search(string).group(1))
        return extracted_value
    except AttributeError:
        pass
    #e.g. <0.05 -> 0.05  |  >72.0 -> 72.0    | ~12 -> 12
    lessthan_roughly_regex = re.compile('([<]|[~]|[>])=?\s*\d+\.*\d*')
    try:
        extracted_value = lessthan_roughly_regex.search(string).group()
        num_regex = re.compile('\d+\.*\d*')
        extracted_value = num_regex.search(extracted_value).group()
        return float(extracted_value)
    except:
        pass
    # e.g. 0.4:0.6 (ratios)
    if ':' in string:
        split = string.split(":")
        try:
            extracted_value = round(float(split[0])/float(split[1]), 3)
            return extracted_value
        except:
            pass
    # e.g. 220 [29] --> 220.0, where citations given, rejecting ab 220 [29] or 7 220 [29]
    if '[' in string and ']' in string:
        try:
            extracted_value = string[:string.index('[')]
            return float(extracted_value)
        except:
            pass
    # e.g. 723 (first peak or other text) --> 723.0
    if '(' in string and ')' in string:
        try:
            extracted_value = string[:string.index('(')]
            return float(extracted_value)
        except:
            pass
    # e.g. '150K' or '150 degC' --> 150.0 but not '1350degC/2h' or 'njn 150 njn'
    try:
        exact_number_regex = re.compile('^(\d+\.?\d*)\s*[a-zA-Z]+$')
        # Using search to find the first occurrence of the pattern in the string
        match = exact_number_regex.search(string)
#         print('Match')
#         print(match)
        
        # If a match is found, extract the numeric part and convert to float
        if match:
            numeric_part = match.group(1)
            extracted_value = float(numeric_part)
            return extracted_value
    except:
        pass
    return None

In [17]:
num_pattern = re.compile(r'\d*\.\d+|\d+')

def num_post_process(num):
    return float(num)

# old get_nums
# def get_nums(table):
#     nums = []
#     for r in table:
#         r_comps, r_nums = [], []
#         for cell in r:
#             # cell_nums = re.findall(num_pattern, re.sub(cons_pattern, ' ', cell))
#             cell_nums = re.findall(num_pattern, cell)
#             r_nums.append(list(set(map(num_post_process, cell_nums))))
#         nums.append(r_nums)
#     # print(f'nums {nums}')
#     return nums

# new get_nums:
def get_nums(table):
    nums = []
    for r in table:
        r_nums = []
        for cell in r:
            num = find_num(cell)
            if num != None:
                cell_nums = [find_num(cell)]
            else:
                cell_nums = []
            r_nums.append(list(set(map(num_post_process, cell_nums))))
        nums.append(r_nums)
    # print(f'nums {nums}')
    return nums

In [18]:
def check_if_cell_can_be_density_header(cell):
    # TODO: better check
    # return "dens" in cell.lower() or "r" in cell.lower()
    return True

def within_tol(num, l, tol):
    # num: value in interglad
    # l: list of values found in a table cell
    # print(f'within_tol {num} {l} {tol}')
    if not l: return False
    # lo, hi = (1 - tol) * num, (1 + tol) * num
    lo, hi = num - tol, num + tol
    return any((lo <= n <= hi for n in l))

In [19]:
pii_types = defaultdict(set)
glass_id_types = defaultdict(set)
index_types = defaultdict(set) # types for rows in comp_property_pii_df
index_to_pii_tidx = {}
# 510 density, 1140 Tg, 2010 ri, 2051 abbe
#prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1123, 1020, 1011, 1118, 1390, 120, 140, 102, 70, 1633, 1305, 1306, 2200, 2209, 2210, 2212, 3188, 1603, 3120]
#prop_names = ['Density', 'GlassTransitionTg',  'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'TStrainP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'ThermalShockResistance', 'TensileStrength', 'CompressiveStrength', 'FlexuralStrength', 'BulkModulus', 'ThermalConductivity', 'DiffusionCoeff', 'ActivationEnergy', 'TransmittanceUV', 'TransmittanceIR', 'TransmittanceVisible', 'TransmittanceSolar', 'DCBreakdownVoltage', 'SpecificHeat', 'LossTangent']
prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 1118, 70, 1306]
prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'BulkModulus', 'ActivationEnergy']
assert len(prop_ids)==len(prop_names)
# prop_ids = [160]
# prop_names = ['FractureToughness']
print(len(prop_ids))
prop_ids_to_index = {prop_id: index for index, prop_id in enumerate(prop_ids)}
index_to_prop_ids = {index: prop_id for prop_id, index in enumerate(prop_ids)}
material_ids_set = set()
prop_count = {prop: 0 for prop in prop_names}
tols = [1e-2]*len(prop_ids)
prop_tables_count = 0
prop_papers_count = 0
total_tables_in_prop_containing_papers = 0
total_tables = 0
total_tuples = 0
have_prop, det_prop = [], []

min_temp = None
max_temp = None
print(index_to_prop_ids)

20
{510: 0, 1140: 1, 2010: 2, 2051: 3, 540: 4, 50: 5, 180: 6, 60: 7, 160: 8, 1014: 9, 1015: 10, 3012: 11, 3174: 12, 1116: 13, 1119: 14, 1020: 15, 1011: 16, 1118: 17, 70: 18, 1306: 19}


In [20]:
print(len(pii_tables))
for prop_idd in prop_id_counts_list:
    if prop_idd not in prop_ids:
        print(prop_idd)
#print(pii_tables['S0022309304005678'])
# pii_tables.keys()
# pii_tables['S0022309304005678'][2]

2536
1603
2210
120
1633
2209
102
1305
1123
2200
2212
1390


In [21]:
def get_dimensions(lst, current_dimensions=None):
    if current_dimensions is None:
        current_dimensions = []

    if not isinstance(lst, list):
        return current_dimensions

    current_dimensions.append(len(lst))

    if len(lst) == 0 or not any(isinstance(item, list) for item in lst):
        return current_dimensions

    return get_dimensions(lst[0], current_dimensions)

In [22]:
#for i, pii in enumerate(list(pii_tables.keys())):
rii_values = []
tol_count = 0
ex_count = 0
# obs_prop = []
# obs_prop_pii = set()
for ind, pii in enumerate(list(pii_tables.keys())):
    counted_once = False
    pii_all_props = comp_property_pii_df[comp_property_pii_df["PII"] == pii]
    if len(pii_all_props) == 0:
        pii_types['dont_have_property'].add(pii)
        for table in pii_tables[pii]:
            total_tables += 1
            table['prop_table'] = False
        continue
    # else
    pii_types['have_property'].add(pii)
    have_prop.append(pii)
    for table in pii_tables[pii]:
        total_tables += 1
        total_tables_in_prop_containing_papers += 1
        r, c = table['num_rows'], table['num_cols']
        # exract numbers from all tables
        table['table_nums'] = get_nums(table['act_table'])
        #print(table['table_nums'])
        table['prop_table'] = False
        table['prop_names'] = set()
        # table['props_header'] = [[False for j in range(table['num_cols'])] for i in range(table['num_rows'])]
        table['prop_orient'] = []
        table['props'] = []
        table['prop_row_col_indexes'] = [] # the indexes of rows and column containing each property
        can_be_props_table = False
        
        density_set_value = set()
        tg_set_value = set()
        rii_set_value = set()
        abbe_set_value = set()
        
        for prop_id in prop_ids:
            table['props'].append([[[] for j in range(table['num_cols'])] for i in range(table['num_rows'])])
            table['prop_row_col_indexes'].append([])
            pii_props = pii_all_props[pii_all_props["PROP_ID"] == prop_id]
            #print(f'Length of pii_props   {pii_props.shape} for {prop_id}')
            #display(pii_props["PROP_FIG"])
            prop_index = prop_ids_to_index[prop_id]
            prop_name = prop_names[prop_index]
            tol = tols[prop_index]
            # check if a header has density or r
            # for i in range(r):
            #     for j in range(c):
            #         if check_if_cell_can_be_density_header(table['act_table'][i][j]):
            #             table['props_header'][i][j] = True
            #             can_be_props_table = True
            # if not can_be_props_table:
            #     continue
            # now find all properties in the all tables
            
            for index, db_row in pii_props.iterrows():
                db_row = db_row[db_row != 0.0]
                # if i == 10:
                #     print(index, db_row)
                # match density values from db_row
                # TODO: unit conversion if required
                ###density_value = db_row["PROP_FIG"]
                # allow at max 2 matches for this density_value in the table
                #max_match_count = max(3, int(len(prop_name)/2))
                max_match_count = 4
                #max_exact_count = max(6, len(prop_name))
                max_exact_count = 8
                match_count = 0
                exact_count = 0
                
                for i in range(r):
                    if match_count == max_match_count or exact_count == max_exact_count:
                        break
                    for j in range(c):
                        # print(prop_name)
                        if match_count == max_match_count or exact_count == max_exact_count:
                            break
                        density_value = db_row["PROP_FIG"]
                        if prop_name == "Density":
                            density_set_value.add(density_value)
                            # print("yo")
                            # nums_times_1000 = [1000*v for v in table['table_nums'][i][j]]
                            # nums_by_1000 = [v/1000 for v in table['table_nums'][i][j]]
                            temp_tol = tol
                            unit = None
                            unit_match_type = 0 # 0 if for matching exact value in interglad, 1 is for matching interglad value/1000 and 2 for matching interglad_value*1000
                            if len(table['table_nums'][i][j]) != 0 and table['table_nums'][i][j][0] < 100:
                                unit = "gm/cm3"
                            elif len(table['table_nums'][i][j]) != 0 and table['table_nums'][i][j][0] >= 100:
                                unit = "kg/m3"
                            if density_value >= 100 and len(table['table_nums'][i][j]) != 0 and table['table_nums'][i][j][0] < 100:
                                density_value = density_value/1000
                                unit_match_type = 1
                            elif density_value < 100 and len(table['table_nums'][i][j]) != 0 and table['table_nums'][i][j][0] >= 100:
                                density_value = density_value*1000
                                temp_tol = tol*1000
                                unit_match_type = 2
                            
#                             if density_value == 2.52:
#                                 if within_tol(density_value, table['table_nums'][i][j], temp_tol):
#                                     print('Found here .............. ')
                            
                            # if within_tol(density_value, table['table_nums'][i][j], tol) or within_tol(density_value, nums_times_1000, tol*1000) or within_tol(density_value, nums_by_1000, tol):
                            if within_tol(density_value, table['table_nums'][i][j], 0):
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit, unit_match_type)) # this table cell potentially contains the property value corresponding to this db row
                                exact_count += 1
                                ex_count += 1
#                                 if density_value == 2.52:
#                                     print('appended')
#                                     print(table['props'][-1])
                            elif within_tol(density_value, table['table_nums'][i][j], temp_tol):
                                # table['props'][-1][i][j].append(index)
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit, unit_match_type)) # this table cell potentially contains the property value corresponding to this db row
                                match_count += 1
                                tol_count += 1
                                
                        elif prop_name == "GlassTransitionTg":
                            tg_set_value.add(density_value)
                            unit = None
                            unit_match_type = None # 0 for matching interglad value, 1 for matching interglad value-273, 2 for matching interglad value+273
                            nums_minus_273 = [v-273 for v in table['table_nums'][i][j]]
                            nums_plus_273 = [v+273 for v in table['table_nums'][i][j]]
                            # unit = 'kelvin'
                            # unit = 'celcius'
                            
                            
                            if within_tol(density_value, table['table_nums'][i][j], 0):    
#                                 if within_tol(density_value, table['table_nums'][i][j], 0):
#                                     unit_match_type = 0
#                                 elif within_tol(density_value, nums_minus_273, 0):
#                                     unit_match_type = 2
#                                 else:
#                                     unit_match_type = 1
                                unit_match_type = 0
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit, unit_match_type))
                                exact_count += 1
                                ex_count += 1
                                
                            elif within_tol(density_value, table['table_nums'][i][j], tol) or within_tol(density_value, nums_minus_273, tol) or within_tol(density_value, nums_plus_273, tol):
                                if within_tol(density_value, table['table_nums'][i][j], tol):
                                    unit_match_type = 0
                                elif within_tol(density_value, nums_minus_273, tol):
                                    unit_match_type = 2
                                else:
                                    unit_match_type = 1
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit, unit_match_type))
                                match_count += 1
                                tol_count += 1
                                
                                
                        elif prop_name == "RefractiveIndex":
                            rii_set_value.add(density_value)
                            unit = ''
                            
                            if within_tol(density_value, table['table_nums'][i][j], 0):
                                # table['props'][-1][i][j].append(index) # this table cell potentially contains the property value corresponding to this db row
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit))
                                exact_count += 1
                                ex_count += 1
                                rii_values.append(index)
                                
                            elif within_tol(density_value, table['table_nums'][i][j], tol):
                                # table['props'][-1][i][j].append(index) # this table cell potentially contains the property value corresponding to this db row
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit))
                                match_count += 1
                                tol_count += 1
                                rii_values.append(index)
                                
                                
                        else:
                            abbe_set_value.add(density_value)
                            unit = ''
                            
                            if within_tol(density_value, table['table_nums'][i][j], 0):
                                # table['props'][-1][i][j].append(index) # this table cell potentially contains the property value corresponding to this db row
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit))
                                exact_count += 1
                                ex_count += 1
                            
                            elif within_tol(density_value, table['table_nums'][i][j], tol):
                                # table['props'][-1][i][j].append(index) # this table cell potentially contains the property value corresponding to this db row
                                table['props'][-1][i][j].append((index, table['table_nums'][i][j][0], unit))
                                match_count += 1
                                tol_count += 1

            # apply the constraint that we can match the interglad value after only one kind of operation for all cells of table
            
#             print('Checking.......11111........')
#             print(table['props'][-1])
            
            if prop_name == "Density" or prop_name == "GlassTransitionTg":
                unit_match_type_counts = {i: 0 for i in [0, 1, 2]}
                for i in range(r):
                    for j in range(c):
                        for tuple in table['props'][-1][i][j]:
                            #print(f'tuple ==== {tuple}')
#                             if tuple[1]==2.52:
#                                 print('LOOK HERE.....')
#                                 print(tuple)
                            unit_match_type_counts[tuple[-1]] += 1
                #print(f'unit_match_type_counts   ==  {unit_match_type_counts}')
            
                unit_match_type = max(unit_match_type_counts, key=unit_match_type_counts.get)
                for i in range(r):
                    for j in range(c):
                        copy_props = table['props'][-1][i][j][:]
                        table['props'][-1][i][j] = []
                        for tuple in copy_props:
                            if tuple[-1] == unit_match_type:
                                table['props'][-1][i][j].append((tuple[0], tuple[1], tuple[2]))
                                
#             print('Checking...............') #imp fpr understanding whats going on
#             print(table['props'][-1])
                        
            
            
            threshold = 0.3 # TODO: can change this
            row_col_scores = [0 for i in range(r+c)] #threshold checking for rows+cols, as we dont know the orientation
            row_col_abs_scores = [0 for i in range(r+c)]
            for i in range(r):
                count = 0
                for j in range(c):
                    if len(table['props'][-1][i][j]) != 0:
                        count += 1
                # needs a cell in this row or col which can be prop header
                # has_prop_header = False
                # for j in range(c):
                #     if table['props_header'][-1][i][j]:
                #         has_prop_header = True
                #         break
                # if not has_prop_header:
                #     continue
                # set score in list
                # if (count/c) > threshold:
                row_col_scores[i] = (count/c)
                row_col_abs_scores[i] = count
            for j in range(c):
                count = 0
                for i in range(r):
                    if len(table['props'][-1][i][j]) != 0:
                        count += 1
                # needs a cell in this row or col which can be prop header

                row_col_scores[r+j] = (count/r)
                row_col_abs_scores[r+j] = count
            if ((max(row_col_abs_scores[:r]) <= max(row_col_abs_scores[r:])) and max(row_col_scores[r:]) < threshold) or ((max(row_col_abs_scores[:r]) > max(row_col_abs_scores[r:])) and max(row_col_scores[:r]) < threshold):
                # print(row_col_scores)
                continue
            table['prop_table'] = True
            table['prop_names'].add(prop_name)
            if not counted_once:
                counted_once = True
                prop_papers_count += 1
                det_prop.append(pii)
            # if max(row_col_scores[:r]) < max(row_col_scores[r:]): # orientation is col
            if max(row_col_abs_scores[:r]) <= max(row_col_abs_scores[r:]): # orientation is col
                for j in range(c):
                    if row_col_scores[j+r] >= threshold:
#                         if pii == "S002230939900681X" and table['t_idx'] == 1:
#                             print("yo assigning orient col with col", j)
                        # max_index = row_col_scores.index(max(row_col_scores))
                        max_index = j+r
                        table['prop_orient'].append('col')
                        table['prop_row_col_indexes'][-1].append(max_index - r)
                        for i in range(r):
                            # for found_index, val, unit in table['props'][-1][i][max_index - r]:
                            if len(table['props'][-1][i][max_index - r]) != 0:
                                total_tuples += 1
                                found_index, val, unit = table['props'][-1][i][max_index - r][0]
                                index_types['found_in_paper'].add(found_index)
                                index_types[prop_name].add(found_index)
                                prop_count[prop_name] += 1
#                                 if prop_name == 'FractureToughness':
#                                     obs_prop.append(((val, comp_property_pii_df['PROP_FIG'][found_index]), pii))
#                                     obs_prop_pii.add(pii)
                                index_to_pii_tidx[found_index] = (pii, table['t_idx'])
                                glass_id_types['found_in_paper'].add(comp_property_pii_df.loc[found_index]["GLASS_ID"])
                                material_ids_set.add(comp_property_pii_df.loc[found_index]["GLASS_ID"])
            else:
                for i in range(r):
                    if row_col_scores[i] >= threshold:
                        max_index = i
                        table['prop_orient'].append('row')
                        table['prop_row_col_indexes'][-1].append(max_index)
                        for j in range(c):
                            # for found_index, val, unit in table['props'][-1][max_index][j]:
                            if len(table['props'][-1][max_index][j]) != 0:
                                total_tuples += 1
                                found_index, val, unit = table['props'][-1][max_index][j][0]
                                index_types['found_in_paper'].add(found_index)
                                index_types[prop_name].add(found_index)
                                prop_count[prop_name] += 1
#                                 if prop_name == 'FractureToughness':
#                                     obs_prop.append(((val, comp_property_pii_df['PROP_FIG'][found_index]), pii))
#                                     obs_prop_pii.add(pii)
                                index_to_pii_tidx[found_index] = (pii, table['t_idx'])
                                glass_id_types['found_in_paper'].add(comp_property_pii_df.loc[found_index]["GLASS_ID"])
                                material_ids_set.add(comp_property_pii_df.loc[found_index]["GLASS_ID"])

        if len(table['prop_orient']) != 0:
            if not len(set(table['prop_orient'])) == 1:
                print("INVALID", pii, table['t_idx'])
                print(table['caption'])
                print(row_col_scores)
                display(pd.DataFrame(table['act_table']))
                print(table['table_nums'])
                print(table['prop_table'])
                print(table['prop_names'])
                print(table['prop_orient'])
                print(table['props'])
                print(table['prop_row_col_indexes']) ## will be taken care in the upcoming pipeline
        if table['prop_table']:
            prop_tables_count += 1


#         print(f'density_set_value  =   {density_set_value}')
#         print(f'tg_set_value  =   {tg_set_value}')
#         print(f'rii_set_value  =   {rii_set_value}')
#         print(r,c)
#         #print(f'abbe_set_value  =   {abbe_set_value}')
#         print()
#         dimensions = get_dimensions(table['props'])
#         print(dimensions)
#         print(table['props'])
#         #print(table['props'])


print(f'min temp {min_temp} max temp {max_temp}')
            
print(f"Out of {len(pii_tables)} piis,")
for key in pii_types.keys():
    print(len(pii_types[key]), key)
# count = comp_property_pii_df["GLASS_ID"].nunique()
# print(f"Out of {count} glass ids in database,")
# for key in glass_id_types.keys():
#     print(len(glass_id_types[key]), key)
print(f"Found properties in {prop_tables_count} tables from {prop_papers_count} papers")
print(f"Total tables in prop containing papers {total_tables_in_prop_containing_papers}")
print(f"Total tables {total_tables} tables")
print(f"Out of {len(comp_property_pii_df)} property rows in database obtained from paper+table props,")
# for key in index_types.keys():
#     print(len(index_types[key]), key, f'({len(index_types[key])/len(comp_property_pii_df)*100}%)')
    
print()

summ_prop = 0
for det_pr in det_prop:
    comm_pro = comp_property_pii_df[comp_property_pii_df["PII"] == det_pr]
    summ_prop += len(comm_pro)
    
# print(f"Out of {summ_prop} property rows in database obtained from table props,")
# for key in index_types.keys():
#     print(len(index_types[key]), key, f'({len(index_types[key])/summ_prop*100}%)')
    
summ_pro = 0
for det_pr in det_prop:
    comm_pro = comp_property_pii_df[comp_property_pii_df["PII"] == det_pr]
    for p_id in prop_ids:
        comm_proo = comm_pro[comm_pro['PROP_ID'] == p_id]
        summ_pro += len(comm_proo)
        
print(f"Out of {summ_prop} property rows in database obtained from table props,")
for key in index_types.keys():
    print(len(index_types[key]), key, f'({len(index_types[key])/summ_prop*100}%)')

print()
print(f"No of property cells for each property")
for key in prop_count.keys():
    print(prop_count[key], key)    

print(f'TOTAL TUPLES {total_tuples}')
print(f'DISTINCT MATERIALS {len(material_ids_set)}')

INVALID S0022309303004563 1
Significant temperatures from DTA analysis
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


,0,1,2,3,4,5,6
0,Sample,T g (degC),T x (degC),T c1 (degC),T c2 (degC),T c3 (degC),T L (degC)
1,Fine,393+-5,438+-5,486+-5,589+-5,668+-5,724+-5
2,Coarse,394+-5,442+-5,484+-5,615+-5,-,698+-5


[[[], [], [], [], [], [], []], [[], [393.0], [438.0], [486.0], [589.0], [668.0], [724.0]], [[], [394.0], [442.0], [484.0], [615.0], [], [698.0]]]
True
{'CrystallizationTemp', 'GlassTransitionTg', 'LiquidusTemperature'}
['col', 'row', 'row', 'col']
[[[[], [], [], [], [], [], []], [[], [], [], [], [], [], []], [[], [], [], [], [], [], []]], [[[], [], [], [], [], [], []], [[], [(10030, 393.0, None)], [], [], [], [], []], [[], [], [], [], [], [], []]], [[[], [], [], [], [], [], []], [[], [], [], [], [], [], []], [[], [], [], [], [], [], []]], [[[], [], [], [], [], [], []], [[], [], [], [], [], [], []], [[], [], [], [], [], [], []]], [[[], [], [], [], [], [], []], [[], [], [], [], [], [], []], [[], [], [], [], [], [], []]], [[[], [], [], [], [], [], []], [[], [], [], [], [], [], []], [[], [], [], [], [], [], []]], [[[], [], [], [], [], [], []], [[], [], [], [], [], [], []], [[], [], [], [], [], [], []]], [[[], [], [], [], [], [], []], [[], [], [], [], [], [], []], [[], [], [], [], [], [], [

,0,1,2,3,4
0,[100x[Cl-]/[Cl-]+[F-]] (anion%),Density (g/cm3),Thermal characteristics,Thermal characteristics,Refractive index
1,[100x[Cl-]/[Cl-]+[F-]] (anion%),Density (g/cm3),T g (degC),T x (degC),Refractive index
2,0,4.60,307+-1,392+-1,1.514+-0.001
3,18,4.15,241+-5,285+-5,1.546+-0.005


[[[], [], [], [], []], [[], [], [], [], []], [[0.0], [4.6], [307.0], [392.0], [1.514]], [[18.0], [4.15], [241.0], [285.0], [1.546]]]
True
{'RefractiveIndex', 'GlassTransitionTg', 'Density'}
['col', 'row', 'col']
[[[[], [], [], [], []], [[], [], [], [], []], [[], [(3872, 4.6, 'gm/cm3')], [], [], []], [[], [(3884, 4.15, 'gm/cm3')], [], [], []]], [[[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []], [[], [], [(3886, 241.0, None)], [(3878, 285.0, None)], []]], [[[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], [(3875, 1.514, ''), (3879, 1.514, '')]], [[], [], [], [], [(3887, 1.546, '')]]], [[[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []]], [[[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []]], [[[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []]], [[[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []], [[], [], [], [], []]], [[[], [], [], [], []], 

,0,1,2,3,4,5,6,7,8
0,Sample,T g,T on set,T 1peak,DH 1,T 2peak,DH 2,T 3peak,DH 3
1,As-cast,641,698,710,-24,734,-45,781,-12
2,2% rolled,637,696,710,-24,731,-42,780,-11
3,18% rolled,630,692,702,-24,730,-40,775,-11


[[[], [], [], [], [], [], [], [], []], [[], [641.0], [698.0], [710.0], [-24.0], [734.0], [-45.0], [781.0], [-12.0]], [[], [637.0], [696.0], [710.0], [-24.0], [731.0], [-42.0], [780.0], [-11.0]], [[], [630.0], [692.0], [702.0], [-24.0], [730.0], [-40.0], [775.0], [-11.0]]]
True
{'CrystallizationTemp', 'GlassTransitionTg'}
['col', 'row', 'row', 'row']
[[[[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []]], [[[], [], [], [], [], [], [], [], []], [[], [(12936, 641.0, None)], [], [], [], [], [], [], []], [[], [(12941, 637.0, None)], [], [], [], [], [], [], []], [[], [(12946, 630.0, None)], [], [], [], [], [], [], []]], [[[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []]], [[[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []], [[], [], [], [], [], [], [], [], []], [[], [], [

In [23]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [24]:
# save annotations
# root_dir = os.getcwd()
# table_dir = os.path.join(root_dir, 'table_data')
# assert os.path.exists(root_dir)
# if not os.path.exists(table_dir):
#     os.mkdir(table_dir)

prop_table_count = 0
row_labels_absent = 0
col_labels_absent = 0
prop_papers_count = 0
all_train_data_pre = []
for i, pii in enumerate(list(pii_tables.keys())):
    counted_once = False
    for table in pii_tables[pii]:
        if table['prop_table']:
            r, c = table['num_rows'], table['num_cols']
            if not 'row_label' in table:
                row_labels_absent += 1
                table['row_label'] = [0 for i in range(r)]
            if not 'col_label' in table:
                col_labels_absent += 1
                table['col_label'] = [0 for i in range(c)]
            if not table["comp_table"]:
                table['row_label'] = [0 for i in range(r)]
                table['col_label'] = [0 for i in range(c)]
            # annotate glass ids
            if sum(table['gid_row_label']) == 1:
                table['row_label'][table['gid_row_label'].index(1)] = 3
            elif sum(table['gid_col_label']) == 1:
                table['col_label'][table['gid_col_label'].index(1)] = 3
            # annotate density row cols
            if not len(set(table['prop_orient'])) == 1:
#                 print("Different orientations of table for different properties")
#                 print("INVALID", pii, table['t_idx'])
                # print(table['caption'])
                # print(row_col_scores)
                # display(pd.DataFrame(table['act_table']))
                # print(table['table_nums'])
                # print(table['prop_table'])
                # print(table['prop_names'])
                # print(table['prop_orient'])
                # print(table['props'])
                # print(table['prop_row_col_indexes'])
                continue
            # if len(table['prop_names']) > len(table['prop_row_col_indexes'][prop_index]):
            #     print(f'len(table["prop_names"]) len(table["prop_row_col_indexes"][prop_index]) {len(table["prop_names"])} {len(table["prop_row_col_indexes"][prop_index])}')
            #     print("INVALID", pii, table['t_idx'])
            #     print(f'comp_table {table["comp_table"]}')
            #     display(pd.DataFrame(table['act_table']))
            #     print(f'table["prop_names"] {table["prop_names"]}')
            #     print(f'table["prop_orient"] {table["prop_orient"]}')
            #     print(f'table["prop_row_col_indexes"] {table["prop_row_col_indexes"]}')
            #     # assert False
            orient = table['prop_orient'][0]
            for pi, prop_name in enumerate(table['prop_names']):
                prop_index = prop_names.index(prop_name)
                if orient == 'row':
                    for index in table['prop_row_col_indexes'][prop_index]:
#                         if table['row_label'][index] != 0:
#                             print("Multiple labels for a row")
#                             print("INVALID", pii, table['t_idx'])
#                             print(f'comp_table {table["comp_table"]}')
#                             print(f'Found {table["row_label"][index]} at row {index}, wanted to annotate {prop_index+4}')
#                             print(f'row_label {table["row_label"]}')
#                             print(f'col_label {table["col_label"]}')
#                             # print(table['caption'])
#                             # print(row_col_scores)
#                             display(pd.DataFrame(table['act_table']))
                            # print(table['table_nums'])
                            # print(table['prop_table'])
                            # print(table['prop_names'])
                            # print(table['prop_orient'])
                            # print(table['props'])
                            # print(table['prop_row_col_indexes'])
                            # assert False
                        table['row_label'][index] = prop_index+4
                elif orient == 'col':
                    for index in table['prop_row_col_indexes'][prop_index]:
#                         if table['col_label'][index] != 0:
#                             print("Multiple labels for a col")
#                             print("INVALID", pii, table['t_idx'])
#                             print(f'comp_table {table["comp_table"]}')
#                             print(f'Found {table["col_label"][index]} at col {index}, wanted to annotate {prop_index+4}')
#                             print(f'row_label {table["row_label"]}')
#                             print(f'col_label {table["col_label"]}')
#                             # print(table['caption'])
#                             # print(row_col_scores)
#                             display(pd.DataFrame(table['act_table']))
                            # print(table['table_nums'])
                            # print(table['prop_table'])
                            # print(table['prop_names'])
                            # print(table['prop_orient'])
                            # print(table['props'])
                            # print(table['prop_row_col_indexes'])
                            # assert False
                        table['col_label'][index] = prop_index+4
                else:
                    print("Unknown error: orient both not row and col for prop_table")
                    assert False
            table['glass_id_type'] = None
            prop_table_count += 1
            if not counted_once:
                counted_once = True
                prop_papers_count += 1
            tid = table['t_idx']
#             save_dir = os.path.join(save_data, pii+'__'+str(tid))
#             if not os.path.exists(save_dir):
#                 os.mkdir(save_dir)
            #pickle.dump(table, open(os.path.join(save_dir, "init.pkl"), 'wb'))
            all_train_data_pre.append(table)
print(f'Saved {prop_table_count} tables from {prop_papers_count} papers')
pickle.dump(all_train_data_pre, open(os.path.join(table_dir, "all_train_data_pre.pkl"), 'wb'))

Saved 425 tables from 336 papers
